In [ ]:
import numpy as np
import glob
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from osgeo import gdal, osr, ogr
from shapely import from_wkt

import subprocess
import os
import re
import time
import scipy
import pandas as pd
import sys
from pandas import DataFrame
from IPython.display import Image
import sklearn
from sklearn import metrics
from matplotlib.pyplot import figure
from matplotlib.colors import ListedColormap
from ipywidgets import interactive
from rasterio.plot import show_hist
from rasterio.enums import ColorInterp
from rasterio.enums import Resampling
import datetime
from datetime import date, timedelta
from datetime import datetime, UTC
import math
os.environ['GDAL_DATA'] =  os.popen('gdal-config --datadir').read().rstrip() #resolves gdal directory value error

from pathlib import Path
import io
import rasterio
import zipfile
import h5py
from pyproj import CRS
import geopandas as gpd
import copy
import asf_search as asf
import earthaccess
import json
from getpass import getpass
import requests

from IPython.display import display, HTML
from tqdm.notebook import tqdm
gdal.UseExceptions()

In [ ]:
work_dir=os.getcwd()
wkt_area='POLYGON((-100.063 19.849, -100.063 19.414, -99.723 19.414, \
-99.723 19.849, -100.063 19.849))'  #Atlacomulco
#print(work_dir)
dataset_name='NISAR'
product='GCOV'
polarizations='HH+HV'

EPSG_new='EPSG:4326'
target_x_res=100.0 #new resolution= 100 m
target_y_res=100.0 #new resolution= 100 m

In [ ]:
def subset_NISAR_processing(input_file, output_file, EPSG_new, target_x_res, target_y_res, wkt_area):
    ds_rescaled=gdal.Warp('temp_rescaled.tif',input_file,format='GTiff',
         xRes=target_x_res, yRes=target_y_res,resampleAlg='Bilinear')  #Going from the original resolution (in m) to new resolution (in m)
    ds_ReProj=gdal.Warp('temp_ReProj.tif',ds_rescaled, format='GTiff',
                            srcSRS='EPSG:32614', dstSRS=EPSG_new) #Change from UTM North (32614) to EPSG:4326 (WGS:84)
    input_file = None

    #Clip using wkt 
    geometry=from_wkt(wkt_area)
    bounds = geometry.bounds
    xmin, ymin, xmax, ymax = bounds
    gdal.Warp(output_file,ds_ReProj,format='GTiff',
              outputBounds=[xmin,ymin,xmax,ymax])
    ds_rescaled = ds_ReProj = None
    
    try:
        os.remove('temp_rescaled.tif')
        os.remove('temp_ReProj.tif')
    except OSError as e:
        print(f"Error: {e}")
  

In [ ]:
def mosaic_generation(num_images,end_time_str):

    list_HHHH=glob.glob("subset_*PR*HHHH.tif")
    list_HVHV=glob.glob("subset_*PR*HVHV.tif")
    if len(list_HHHH) == len(list_HVHV):
        if num_images>1:
            gdal.Warp('mosaic_HHHH.tif',list_HHHH,format='GTiff')
            destination_file = 'subset_NISAR_L2_GCOV_'+end_time_str[0:10]+'_HHHH.tif'
            command = f"mv mosaic_HHHH.tif {destination_file}"
            os.system(command)
                
            gdal.Warp('mosaic_HVHV.tif',list_HVHV,format='GTiff')
            destination_file = 'subset_NISAR_L2_GCOV_'+end_time_str[0:10]+'_HVHV.tif'
            command = f"mv mosaic_HVHV.tif {destination_file}"
            os.system(command)
        else:

            destination_file = 'subset_NISAR_L2_GCOV_'+end_time_str[0:10]+'_HHHH.tif'
            command = f"mv {list_HHHH[0]} {destination_file}"
            os.system(command)
            destination_file = 'subset_NISAR_L2_GCOV_'+end_time_str[0:10]+'_HVHV.tif'
            command = f"mv {list_HVHV[0]} {destination_file}"
            os.system(command)
    else:
        print(f"Error creating the mosaic for {end_time_str}")
        

In [ ]:
session=asf.ASFSession()

In [ ]:
start_year=2025
end_year=2026
start_date = date(start_year, 10, 15)   #start_date = date(start_year, 1, 1)
end_date = date(end_year, 3, 10) #end_date = date(end_year, 12, 31)

download_path=os.path.join(work_dir,'tiles')
if(not os.path.isdir(download_path)):
    os.mkdir(download_path)
    
os.chdir(download_path) 
interval = timedelta(days=4) # 4-day increment
while start_date <= end_date:
    date1=start_date
    #[year,month,day] = [start_date.year,start_date.month,start_date.day]
    start_date += interval # 4-day increment
    date2=start_date
    
    start_time_str=date1.strftime("%Y-%m-%dT%H:%M:%SZ")
    end_time_str=date2.strftime("%Y-%m-%dT%H:%M:%SZ")
    print(start_time_str,end_time_str)
    try:
        results = asf.geo_search(dataset=dataset_name,
                             processingLevel=product,
                             mainBandPolarization=polarizations,
                             intersectsWith=wkt_area,
                             start=start_time_str,
                             end=end_time_str)
#                            maxResults=1)
        print(f"Found {len(results)} results ")

        if results:
            #print(f"Downloading images to: {download_path}")
            results.download(path=download_path, session=session, processes=4)

            list_h5=glob.glob("*.h5")
            for file_name in list_h5:
                #print(file_name)
                input_file_HHHH='NETCDF:"'+file_name+'"://science/LSAR/GCOV/grids/frequencyA/HHHH'
                input_file_HVHV='NETCDF:"'+file_name+'"://science/LSAR/GCOV/grids/frequencyA/HVHV'
                output_file_HHHH='subset_'+file_name[0:-4]+'_HHHH'+'.tif'
                output_file_HVHV='subset_'+file_name[0:-4]+'_HVHV'+'.tif'
                
                #print(input_file_HHHH)
                #print(output_file_HHHH)
                subset_NISAR_processing(input_file_HHHH, output_file_HHHH, EPSG_new, target_x_res, target_y_res, wkt_area)
                subset_NISAR_processing(input_file_HVHV, output_file_HVHV, EPSG_new, target_x_res, target_y_res, wkt_area)

            mosaic_generation(len(list_h5),end_time_str)
           
            for file_name in list_h5:     
                try:
                    os.remove(file_name)
                except OSError as e:
                    print(f"Error: {e}")

            list_remove = glob.glob("subset_*PR*.tif")
            for file_name in list_remove:     
                try:
                    os.remove(file_name)
                except OSError as e:
                    print(f"Error: {e}")
            
        else:
            print("No result")
    except Exception as e:
        print(f"Error duirng asf_search: {e}")
        